In [9]:
!nvidia-smi
!git clone https://github.com/KUO-WEL/CA_Final_Project.git
%cd CA_Final_Project

Tue Jun  2 08:10:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
%%writefile main.cu
#include <iostream>
#include <cuda_runtime.h>
#include "include/dataset.h" // 引入你的測資

// ========================================================
// CUDA Kernel: 每個 Thread 負責計算一個視窗位置的 Cross-Correlation
// ========================================================
__global__ void cross_correlation_kernel(const float* d_rx, const float* d_preamble, float* d_result, int num_windows) {
    // 1. 計算當前 Thread 的全域索引 (Global Index)
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    // 2. 宣告 Shared Memory 用來存放 Preamble
    // 因為每個 block 裡的 thread 都需要重複讀取同一段 Preamble，放進 Shared Memory 可以大幅節省 Global Memory 頻寬。
    __shared__ float s_preamble[PREAMBLE_LEN];

    // 3. 協同載入 (Collaborative Loading): 讓 Block 裡面的 Threads 一起把 Preamble 搬進 Shared Memory
    int local_tid = threadIdx.x;
    if (local_tid < PREAMBLE_LEN) {
        s_preamble[local_tid] = d_preamble[local_tid];
    }

    // 確保所有 Thread 都把資料搬完了，才能往下走
    __syncthreads();

    // 4. 執行相關性計算 (MAC)
    if (tid < num_windows) {
        float sum = 0.0f;
        for (int j = 0; j < PREAMBLE_LEN; ++j) {
            // 注意：這裡讀取 Preamble 是從極速的 Shared Memory 讀取！
            sum += s_preamble[j] * d_rx[tid + j];
        }
        d_result[tid] = sum;
    }
}

// ========================================================
// Host 主程式
// ========================================================
int main() {
    int num_windows = RX_LEN - PREAMBLE_LEN + 1;
    size_t rx_bytes = RX_LEN * sizeof(float);
    size_t preamble_bytes = PREAMBLE_LEN * sizeof(float);
    size_t result_bytes = num_windows * sizeof(float);

    // 取得 Pattern 0 的起始指標 (Host 端)
    const float* h_rx = &RX_SIGNALS[0 * RX_LEN];
    float* h_result = new float[num_windows];

    // 1. 配置 GPU (Device) 記憶體
    float *d_rx, *d_preamble, *d_result;
    cudaMalloc((void**)&d_rx, rx_bytes);
    cudaMalloc((void**)&d_preamble, preamble_bytes);
    cudaMalloc((void**)&d_result, result_bytes);

    // 2. 將資料從 CPU (Host) 複製到 GPU (Device)
    cudaMemcpy(d_rx, h_rx, rx_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_preamble, PREAMBLE, preamble_bytes, cudaMemcpyHostToDevice);

    // 3. 設定 Thread 與 Block 的維度
    int threads_per_block = 256; // 專題建議你可以調整這個數值來觀察效能差異
    int blocks_per_grid = (num_windows + threads_per_block - 1) / threads_per_block;

    // [建立 CUDA 計時器]
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // 4. 啟動 Kernel
    cudaEventRecord(start);
    cross_correlation_kernel<<<blocks_per_grid, threads_per_block>>>(d_rx, d_preamble, d_result, num_windows);
    cudaEventRecord(stop);

    // 5. 將運算結果從 GPU 複製回 CPU
    cudaMemcpy(h_result, d_result, result_bytes, cudaMemcpyDeviceToHost);
    cudaEventSynchronize(stop);

    // 6. 計算時間與驗證結果
    float ms;
    cudaEventElapsedTime(&ms, start, stop);

    float max_corr = -1e9;
    int detected_idx = -1;
    for (int i = 0; i < num_windows; ++i) {
        if (h_result[i] > max_corr) {
            max_corr = h_result[i];
            detected_idx = i;
        }
    }

    std::cout << "=== Part 4: CUDA SIMT Execution ===" << std::endl;
    std::cout << "Kernel 執行時間: " << ms << " ms" << std::endl;
    std::cout << "偵測到的 Preamble 索引: " << detected_idx << std::endl;

    if (detected_idx == GROUND_TRUTH[0]) {
        std::cout << "=> GPU 驗證成功！" << std::endl;
    } else {
        std::cout << "=> GPU 驗證失敗！" << std::endl;
    }

    // 7. 釋放記憶體
    cudaFree(d_rx);
    cudaFree(d_preamble);
    cudaFree(d_result);
    delete[] h_result;

    return 0;
}

Overwriting main.cu


In [14]:
!nvcc -Xptxas -v main.cu -o main -O2 -arch=sm_75
!./main

ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z24cross_correlation_kernelPKfS0_Pfi' for 'sm_75'
ptxas info    : Function properties for _Z24cross_correlation_kernelPKfS0_Pfi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 66 registers, used 1 barriers, 1024 bytes smem, 380 bytes cmem[0]
ptxas info    : Compile time = 34.007 ms
=== Part 4: CUDA SIMT Execution ===
Kernel 執行時間: 0.1208 ms
偵測到的 Preamble 索引: 1070
=> GPU 驗證成功！


In [17]:
!ncu --set basic ./main

==PROF== Connected to process 12821 (/content/CA_Final_Project/CA_Final_Project/CA_Final_Project/main)
==PROF== Profiling "cross_correlation_kernel" - 0: 0%....50%....100% - 9 passes
=== Part 4: CUDA SIMT Execution ===
Kernel 執行時間: 2092 ms
偵測到的 Preamble 索引: 1070
=> GPU 驗證成功！
==PROF== Disconnected from process 12821
[12821] main@127.0.0.1
  cross_correlation_kernel(const float *, const float *, float *, int) (16, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       581.41
    Elapsed Cycles                cycle        7,916
    Memory Throughput                 %        24.94
    DRAM Throughput                   %         0.90
    Duration                         us        13.60
    L1/